# 1. Silver Layer: Data Cleaning & Validation

This notebook transforms raw Bronze data into clean, structured data.

Key steps:
- Fix inconsistent formats
- Cast data types
- Handle missing values
- Remove outliers
- Deduplicate records
- Apply data quality checks

## 2. Load Bronze Data

Reading raw IoT data from the Bronze Delta table.

In [0]:
from pyspark.sql.functions import col, regexp_replace, when, count, to_timestamp

df = spark.table("main.default.bronze_machine_readings")

display(df.limit(5))
df.printSchema()

machine_id,pressure,temperature,timestamp,vibration
M3,3.45,85,2026-04-12T10:52:22.060563,0.75
M3,4.15,77,2026-04-12T10:52:22.060591,0.12
M1,3.22,78,2026-04-12T10:52:22.060599,0.85
M3,3.17,67,2026-04-12T10:52:22.060613,0.34
M5,3.58,67,2026-04-12T10:52:22.060619,0.29


root
 |-- machine_id: string (nullable = true)
 |-- pressure: double (nullable = true)
 |-- temperature: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- vibration: double (nullable = true)



## 3. Initial Data Profiling

Quick summary statistics to understand data distribution.

In [0]:
display(df.describe())

summary,machine_id,pressure,temperature,timestamp,vibration
count,500,500,448,500,500
mean,null,2.971519999999998,121.1540404040404,null,0.8256200000000001
stddev,null,1.1428467469323906,202.8820172423929,null,1.1038358223238143
min,M1,1.01,60,2026-04-12T10:52:22.060563,0.1
max,M5,5.0,999,2026-04-12T10:52:22.068193,5.0


## 4. Data Cleaning & Transformation

Steps:
- Remove "C" from temperature values
- Cast columns to correct data types
- Convert timestamp to proper format

In [0]:
df_clean = df.withColumn(
    "temperature",
    regexp_replace(col("temperature"), "C", "")
)

df_clean = df_clean.withColumn("temperature", col("temperature").cast("double")) \
    .withColumn("vibration", col("vibration").cast("double")) \
    .withColumn("pressure", col("pressure").cast("double")) \
    .withColumn("timestamp", to_timestamp(col("timestamp")))

## 5. Handle Missing Values

Removing records where temperature is null.

In [0]:
df_clean = df_clean.filter(col("temperature").isNotNull())

## 6. Outlier Removal

Filtering unrealistic sensor values:
- temperature < 200
- vibration < 2

In [0]:
df_clean = df_clean.filter(
    (col("temperature") < 200) &
    (col("vibration") < 2)
)

## 7. Deduplication

Removing duplicate records based on:
- machine_id
- timestamp

In [0]:
df_clean = df_clean.dropDuplicates(["machine_id", "timestamp"])

## 8. Data Quality Checks

Validating:
- No null temperatures
- Valid temperature range (0–150)
- No duplicate records

In [0]:
# Null check
null_count = df_clean.filter(col("temperature").isNull()).count()

# Range check
invalid_temp = df_clean.filter(
    (col("temperature") < 0) | (col("temperature") > 150)
).count()

# Duplicate check
duplicate_count = df_clean.groupBy("machine_id", "timestamp") \
    .count().filter("count > 1").count()

print("Null count:", null_count)
print("Invalid temp:", invalid_temp)
print("Duplicates:", duplicate_count)

Null count: 0
Invalid temp: 0
Duplicates: 0


## 9. Enforce Data Quality

Fail pipeline if critical data issues are detected.

In [0]:
if null_count > 0:
    raise ValueError("Data quality failed: NULL temperature found")

if invalid_temp > 0:
    raise ValueError("Data quality failed: Invalid temperature values")

if duplicate_count > 0:
    raise ValueError("Data quality failed: Duplicate records found")

## 10. Column-Level Null Summary
 
Optional: Overview of null values across all columns.

In [0]:
null_counts = df_clean.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_clean.columns
])

display(null_counts)

machine_id,pressure,temperature,timestamp,vibration
0,0,0,0,0


## 11. Write Silver Table

Saving cleaned data into Silver Delta table.

In [0]:
df_clean.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("main.default.silver_machine_readings")

## 12. Verification

Preview cleaned data and confirm schema.

In [0]:
display(spark.sql("""
SELECT * 
FROM main.default.silver_machine_readings 
LIMIT 10
"""))

df_clean.printSchema()

machine_id,pressure,temperature,timestamp,vibration
M5,3.69,65.0,2026-04-12T10:52:22.060Z,0.39
M2,1.48,61.0,2026-04-12T10:52:22.060Z,0.22
M3,3.07,66.0,2026-04-12T10:52:22.061Z,0.9
M1,2.18,85.0,2026-04-12T10:52:22.061Z,0.85
M2,4.58,74.0,2026-04-12T10:52:22.062Z,0.14
M1,2.01,85.0,2026-04-12T10:52:22.062Z,0.8
M3,1.5,81.0,2026-04-12T10:52:22.062Z,0.33
M4,2.27,90.0,2026-04-12T10:52:22.062Z,0.73
M4,1.25,64.0,2026-04-12T10:52:22.062Z,0.15
M5,1.11,83.0,2026-04-12T10:52:22.062Z,0.94


root
 |-- machine_id: string (nullable = true)
 |-- pressure: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- vibration: double (nullable = true)



In [0]:
print("Final row count:", df_clean.count())

Final row count: 400
